# 🏠 Roof Corrosion Detection — Kaggle GPU Training Notebook

**Purpose:** End-to-end training for roof footprint and corrosion segmentation on Kaggle's free GPU tier (T4 / P100).
**Version control:** Designed for Kaggle's native GitHub integration. Save versions frequently and link to your GitHub repo for full history.

## Pre-flight Checklist
- [ ] Notebook → Settings → Accelerator: **GPU T4** (recommended) or **GPU P100**
- [ ] Notebook → Settings → Internet: **On**
- [ ] Notebook → Settings → Persistence: **Variables and Files**
- [ ] (Optional) Attach dataset(s) under `/kaggle/input/`
- [ ] Link this notebook to GitHub: File → Link to GitHub → authorize → select repo → `kaggle_notebook.ipynb`

> **Tip:** After linking to GitHub, every **Save Version** (File → Save Version) creates a commit on your linked repo. This gives you full version control without leaving Kaggle.

## 1. GPU Detection & Environment Diagnostics

Auto-detects the available GPU (T4, P100, or CPU) and sets the optimal precision and memory configuration.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import torch
import numpy as np

def detect_gpu():
    """Detect Kaggle GPU type and recommend precision."""
    if not torch.cuda.is_available():
        return "cpu", "32-true", None
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_type = "p100" if "P100" in name else "t4" if "T4" in name else "other"
    # A100/H100 = compute 8.0/9.0+ → bf16 supported. T4 = 7.5 → fp16 (more stable).
    precision = "bf16-mixed" if cap[0] >= 8 else "16-mixed" if cap[0] >= 7 else "32-true"
    info = {
        "device_name": name,
        "compute_capability": f"{cap[0]}.{cap[1]}",
        "memory_gb": round(mem_gb, 1),
        "precision": precision,
    }
    return gpu_type, precision, info

GPU_TYPE, PRECISION, GPU_INFO = detect_gpu()
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.version.cuda} | available: {torch.cuda.is_available()}")
if GPU_INFO:
    for k, v in GPU_INFO.items():
        print(f"  {k:20s}: {v}")
print(f"\nRecommended precision: {PRECISION}")

# Reduce memory fragmentation on older GPUs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"

# Reproducibility seeds (set here, applied after installs)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True  # speed over strict reproducibility

print(f"\nSeed set to {SEED}")


## 2. Version Control — Kaggle ↔ GitHub Integration

This notebook is designed to work with **Kaggle's native GitHub integration**. Once linked, every `Save Version` pushes a commit to your repo.

### One-time setup (do this in the Kaggle UI)
1. File → Link to GitHub → authorize Kaggle → pick `your-username/roof_corrosion_mlops`
2. Set file path: `kaggle_notebook.ipynb`
3. Enable **"Always save version on run"** for automatic commits

### Programmatic helpers (optional)
The cells below let you inspect the current notebook hash and write a version manifest to `/kaggle/working/` for traceability.

In [ ]:
# Write a version manifest for every run — useful for MLflow, W&B, or manual tracking
manifest = {
    "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "kaggle_gpu": GPU_TYPE or "cpu",
    "precision": PRECISION,
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda if torch.cuda.is_available() else None,
    "seed": SEED,
    "git_remote": None,  # populated below if available
    "git_commit": None,
}

# If running inside a cloned git repo (e.g. via Kaggle Datasets + GitHub), capture commit info
repo_dir = Path("/kaggle/input/roof-corrosion-mlops")  # if you attach the repo as a dataset
if not repo_dir.exists():
    repo_dir = Path("/kaggle/working/roof_corrosion_mlops")
if repo_dir.exists() and (repo_dir / ".git").exists():
    try:
        manifest["git_remote"] = subprocess.check_output(
            ["git", "-C", str(repo_dir), "remote", "get-url", "origin"],
            text=True, stderr=subprocess.DEVNULL
        ).strip()
        manifest["git_commit"] = subprocess.check_output(
            ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
            text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        pass

manifest_path = Path("/kaggle/working/version_manifest.json")
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"✓ Version manifest written to {manifest_path}")
print(json.dumps(manifest, indent=2))


## 3. Install Pinned Dependencies

Kaggle's base image gets random PyTorch / CUDA bumps. We pin everything to versions verified against T4 (CUDA 12.1/12.4).

> **Note:** If you see `CUDA mismatch` or `segmentation fault` after a Kaggle maintenance window, re-run this cell to refresh to the pinned versions.

In [ ]:
%%capture --no-stderr
# Core ML stack — pinned for Kaggle T4/P100 (CUDA 12.1+, sm_70 / sm_60)
!pip install -q -U \
    "torch==2.4.1" "torchvision==0.19.1" \
    "transformers==4.46.3" \
    "accelerate==1.1.1" \
    "safetensors==0.4.5" \
    "peft==0.13.2" \
    "huggingface-hub==0.26.5" \
    "datasets==3.1.0" \
    "timm==1.0.11" \
    "einops==0.8.0" \
    "albumentations==1.4.21" \
    "pytorch-lightning==2.4.0" \
    "torchmetrics==1.6.0" \
    "rasterio==1.4.2" \
    "opencv-python-headless==4.10.0.84" \
    "mlflow-skinny==2.18.0" \
    "wandb==0.18.7" \
    "onnx==1.17.0" \
    "onnxruntime-gpu==1.20.1" \
    "omegaconf==2.3.0" \
    "tqdm" \
    "segmentation-models-pytorch"

# Clay / TerraTorch — optional; fallback to direct HF load if unavailable
!pip install -q -U "terratorch==1.0.1" || echo "[WARN] terratorch install failed — will use direct HF load"


In [ ]:
# ── Repository Detection + Auto-Clone ────────────────────────────────────────
GITHUB_USER = "khiwniti"          # <-- your GitHub username
REPO_NAME   = "roof_corrosion_mlops"
AUTO_CLONE  = True                # set False if you prefer manual clone

REPO_CANDIDATES = [
    Path("/kaggle/input/roof-corrosion-mlops"),      # if attached as Kaggle Dataset
    Path("/kaggle/working/roof_corrosion_mlops"),    # if git-cloned manually
    Path("/kaggle/working/roof-corrosion-mlops"),    # alt clone name
    Path.cwd(),                                      # if notebook file is inside repo
]

REPO_ROOT = None
for cand in REPO_CANDIDATES:
    if (cand / "ml").exists() and (cand / "ml" / "baseline").exists():
        REPO_ROOT = cand.resolve()
        break

if REPO_ROOT is None and AUTO_CLONE:
    clone_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
    clone_dir = Path(f"/kaggle/working/{REPO_NAME}")
    # Remove stale clone dir so git clone won't complain on re-runs
    if clone_dir.exists():
        !rm -rf {clone_dir}
    print(f"⚠️  Repo not found. Auto-cloning from {clone_url} ...")
    !git clone --depth 1 {clone_url} {clone_dir}
    if (clone_dir / "ml").exists():
        REPO_ROOT = clone_dir.resolve()
        print(f"✓ Cloned successfully. Repo root: {REPO_ROOT}")
    else:
        print(f"✗ Clone failed or repo does not contain expected structure.")
        print(f"  URL tried: {clone_url}")
        print("  Please check:")
        print(f"    1. GITHUB_USER is correct (currently: '{GITHUB_USER}')")
        print("    2. The repo is public (private repos need a PAT token)")
        print("    3. Internet is enabled: Settings -> Internet -> On")

if REPO_ROOT is None:
    print("⚠️  Repository root not found. Falling back to working directory.")
    print("    Training scripts will NOT be available.")
    print("    To fix:")
    print("    • Set AUTO_CLONE = True (default) and re-run this cell, or")
    print(f"    • Clone manually: !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git")
    REPO_ROOT = Path("/kaggle/working")
else:
    print(f"✓ Repo root: {REPO_ROOT}")
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
        print(f"  Added to sys.path")

# Standard Kaggle paths
INPUT_DIR = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORKING_DIR / "runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nPaths:")
print(f"  INPUT_DIR:   {INPUT_DIR}")
print(f"  WORKING_DIR: {WORKING_DIR}")
print(f"  OUTPUT_DIR:  {OUTPUT_DIR}")


In [ ]:
# Try multiple possible repo locations
REPO_CANDIDATES = [
    Path("/kaggle/input/roof-corrosion-mlops"),      # if attached as Kaggle Dataset
    Path("/kaggle/working/roof_corrosion_mlops"),    # if git-cloned manually
    Path("/kaggle/working/roof-corrosion-mlops"),    # alt clone name
    Path.cwd(),                                      # if notebook file is inside repo
]

REPO_ROOT = None
for cand in REPO_CANDIDATES:
    if (cand / "ml").exists() and (cand / "ml" / "baseline").exists():
        REPO_ROOT = cand.resolve()
        break

if REPO_ROOT is None:
    print("⚠️  Repository root not found. Falling back to working directory.")
    print("    If you want training scripts, either:")
    print("    • Attach this repo as a Kaggle Dataset, or")
    print("    • Clone it in a cell: !git clone https://github.com/<user>/roof_corrosion_mlops.git")
    REPO_ROOT = Path("/kaggle/working")
else:
    print(f"✓ Repo root detected: {REPO_ROOT}")
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
        print(f"  Added to sys.path")

# Standard Kaggle paths
INPUT_DIR = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORKING_DIR / "runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nPaths:")
print(f"  INPUT_DIR:   {INPUT_DIR}")
print(f"  WORKING_DIR: {WORKING_DIR}")
print(f"  OUTPUT_DIR:  {OUTPUT_DIR}")


## 5. Data Discovery

List datasets attached under `/kaggle/input` so you can confirm the correct paths before training.

In [ ]:
def discover_datasets(root: Path = INPUT_DIR, max_files: int = 5):
    """Print first N files per subdirectory under root."""
    if not root.exists():
        print(f"{root} does not exist. No attached datasets.")
        return
    for subdir in sorted(root.iterdir()):
        if subdir.is_dir():
            files = list(subdir.rglob("*"))
            files = [f for f in files if f.is_file()]
            print(f"\n📁 {subdir.name} ({len(files)} files)")
            for f in files[:max_files]:
                print(f"   {f.relative_to(subdir)}")
            if len(files) > max_files:
                print(f"   ... and {len(files) - max_files} more")

discover_datasets()


## 6. Training — Stage 1: Roof Footprint Segmentation

Runs the baseline roof detector (`ml/baseline/train_stage1_roof.py`). Override paths via `--data_dir` if your Kaggle dataset uses a different folder name.

In [ ]:
# ── Stage 1: Roof Segmentation ───────────────────────────────────────────────
#
# Edit DATASET_NAME to match your attached Kaggle dataset folder.
# If the repo is attached as a dataset named `roof-corrosion-mlops`, you can leave it as-is.

DATASET_NAME = "roof-corrosion-mlops"  # <-- change to your dataset name
DATA_DIR = INPUT_DIR / DATASET_NAME / "data"

# Verify that the training script exists
if REPO_ROOT is not None:
    stage1_script = REPO_ROOT / "ml" / "baseline" / "train_stage1_roof.py"
    stage1_config = REPO_ROOT / "ml" / "baseline" / "configs" / "stage1_roof.yaml"
else:
    stage1_script = Path()
    stage1_config = Path()

if stage1_script.exists():
    print(f"✓ Stage 1 script found: {stage1_script}")
    print(f"✓ Stage 1 config found: {stage1_config}")
else:
    print(f"✗ Stage 1 script NOT found at {stage1_script}")
    print("  Please attach the repo as a dataset or clone it.")


In [ ]:
# Run Stage 1 training (uncomment to execute)
# Adjust epochs / batch size based on GPU memory.
#
# T4 16GB  → batch_size 4, epochs 50
# P100 16GB → batch_size 4, epochs 50 (slower than T4)

# !python {stage1_script} --config {stage1_config} \
#     --data_dir {DATA_DIR} \
#     --epochs 50 \
#     --batch_size 4 \
#     --out_dir {OUTPUT_DIR / "stage1_roof"}


## 7. Training — Stage 2: Corrosion Segmentation

Runs the baseline corrosion detector (`ml/baseline/train_stage2_corrosion.py`). Uses Focal + Dice loss for class imbalance.

In [ ]:
# ── Stage 2: Corrosion Segmentation ──────────────────────────────────────────

if REPO_ROOT is not None:
    stage2_script = REPO_ROOT / "ml" / "baseline" / "train_stage2_corrosion.py"
    stage2_config = REPO_ROOT / "ml" / "baseline" / "configs" / "stage2_corrosion.yaml"
else:
    stage2_script = Path()
    stage2_config = Path()

if stage2_script.exists():
    print(f"✓ Stage 2 script found: {stage2_script}")
    print(f"✓ Stage 2 config found: {stage2_config}")
else:
    print(f"✗ Stage 2 script NOT found at {stage2_script}")


In [ ]:
# Run Stage 2 training (uncomment to execute)
#
# T4 16GB  → batch_size 4, epochs 80
# P100 16GB → batch_size 2-4, epochs 80

# !python {stage2_script} --config {stage2_config} \
#     --data_dir {DATA_DIR} \
#     --epochs 80 \
#     --batch_size 4 \
#     --out_dir {OUTPUT_DIR / "stage2_corrosion"}


## 8. Advanced — Clay Multi-Task Training

For the Clay v1.5 + ViT-Adapter + Mask2Former multi-task model (ADR-002). This path requires more VRAM; reduce `img_size` and `batch_size` on P100.

> If you encounter OOM, switch the config to `decoder: upernet` and disable ViT-Adapter.

In [ ]:
# ── Clay Multi-Task ────────────────────────────────────────────────────────

if REPO_ROOT is not None:
    multitask_script = REPO_ROOT / "ml" / "train" / "train_multitask.py"
    multitask_config = REPO_ROOT / "ml" / "train" / "configs" / "multitask_clay.yaml"
else:
    multitask_script = Path()
    multitask_config = Path()

if multitask_script.exists():
    print(f"✓ Multi-task script found: {multitask_script}")
    print(f"✓ Multi-task config found: {multitask_config}")
    print("\nTip: On P100 16GB, edit the YAML to set img_size=384, batch_size=2, decoder=upernet")
else:
    print(f"✗ Multi-task script NOT found at {multitask_script}")


## 9. Export Artifacts & Commit Checklist

Before saving a version (and pushing to GitHub), verify that all important artifacts are in `/kaggle/working/`. Kaggle only persists files under `/kaggle/working/` between runs.

In [ ]:
def list_working_tree(root: Path = WORKING_DIR, max_depth: int = 3):
    """Print tree of /kaggle/working up to max_depth."""
    print(f"\n📦 Artifacts in {root}:")
    for p in sorted(root.rglob("*")):
        if p == root:
            continue
        depth = len(p.relative_to(root).parts)
        if depth > max_depth:
            continue
        indent = "  " * (depth - 1)
        if p.is_dir():
            print(f"{indent}📁 {p.name}")
        else:
            size_mb = p.stat().st_size / 1e6
            print(f"{indent}📄 {p.name} ({size_mb:.2f} MB)")

list_working_tree()

# Commit checklist
print("\n" + "="*60)
print("COMMIT CHECKLIST (before File → Save Version)")
print("="*60)
vm_exists = (WORKING_DIR / 'version_manifest.json').exists()
out_exists = any(OUTPUT_DIR.glob('*'))
marker1 = "[x]" if vm_exists else "[ ]"
print(f"{marker1} version_manifest.json exists")
marker2 = "[x]" if out_exists else "[ ]"
print(f"{marker2} Output checkpoints in {OUTPUT_DIR}")
print("[ ] GitHub linked in Kaggle UI  (check File → Link to GitHub)")
print("="*60)


## 10. Quick Reference — Kaggle Commands

| Action | How |
|---|---|
| **Save Version** | File → Save Version → Save & Run All (commits to GitHub if linked) |
| **Link to GitHub** | File → Link to GitHub → authorize → pick repo → set file path |
| **Add Dataset** | Notebook → Add Input → search `roof-corrosion-mlops` → Attach |
| **Change GPU** | Notebook → Settings → Accelerator → GPU T4 / P100 |
| **Download outputs** | Go to Output tab → download `runs/` or zip `/kaggle/working` |
| **Clean working dir** | `!rm -rf /kaggle/working/*` |

---

**Happy training!** If you hit OOM or CUDA errors, lower `batch_size` / `img_size` and re-run from Section 3.